In [0]:
from pyspark.sql import functions as F

CHECKPOINT = "/Volumes/fleetpulse/raw/checkpoints/bronze"
SCHEMA_LOC = "/Volumes/fleetpulse/raw/checkpoints/bronze_schema"

stream = (spark.readStream
    .format("cloudFiles")                                  # Auto Loader
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)       # where inferred schema is tracked over time
    .option("cloudFiles.schemaEvolutionMode", "rescue")    # unknown/mistyped fields -> _rescued_data, never crash
    .load("/Volumes/fleetpulse/raw/landing/")
    .withColumn("_ingest_ts", F.current_timestamp())       # when WE received it (vs event_ts = when it happened)
    .withColumn("_source_file", F.col("_metadata.file_path")))  # lineage: every row traceable to its file

(stream.writeStream
    .option("checkpointLocation", CHECKPOINT)              # streaming state -> exactly-once
    .trigger(availableNow=True)                            # process all pending files, then STOP (quota-friendly)
    .toTable("fleetpulse.bronze.telemetry_raw"))

In [0]:
%sql
SELECT COUNT(*) FROM fleetpulse.bronze.telemetry_raw;                    -- expect ~206K (sum of your batch prints)

In [0]:
%sql
SELECT * FROM fleetpulse.bronze.telemetry_raw LIMIT 5;                   -- eyeball columns incl. _ingest_ts, _source_file

In [0]:
%sql
SELECT COUNT(*) FROM fleetpulse.bronze.telemetry_raw WHERE _rescued_data IS NOT NULL;  -- did anything get rescued?

In [0]:
%sql
SELECT event_id FROM fleetpulse.bronze.telemetry_raw LIMIT 3;  -- column exists, populated